# Lesson 22 - See It, Grab It (Vision + Claw)

In Lesson 13 the robot used the camera to line up with a ball and **push** it.
In Lesson 21 you learned the **claw**. This lesson puts them together:

1. The camera finds a coloured object.
2. The robot turns/strafes until the object is centred.
3. The robot drives forward until the **distance sensor** says it is close.
4. The **claw** picks the object up.

**This lesson needs a robot with the claw** to do the final grab. On a robot
with no claw it will still find and drive up to the object — it just skips the
grab (and tells you).

## 1. Start The Robot

In [ ]:
from lesson_header import *
import time

show_v2_status()
myRobot = bot(base_speed=300)
myRobot.camera.center_all()

## 2. Set the colour you want to pick up

Use a colour your camera is calibrated for (same idea as Lesson 13). Pick ONE
target colour.

In [ ]:
TARGET_COLOUR = "red"

# If you need to set/adjust the profile, do it like in Lesson 13:
# myRobot.vision.set_color_profile(
#     "red", lower_hsv=(0, 120, 120), upper_hsv=(10, 255, 255),
# )

has_claw = myRobot.arm.available
print("Claw on this robot:", has_claw)
if has_claw:
    myRobot.arm.ready()   # open claw, arm at middle height, ready to grab

## 3. Check your eyes and your ruler

- `target_position` tells you if the object is **left / center / right / lost**.
- `distance_cm` tells you **how far** the nearest thing is, in centimetres.

Run this with the object in view to see both at once.

In [ ]:
decision = myRobot.vision.target_position(TARGET_COLOUR, deadzone=50, show=True)
print("direction:", decision["direction"])
print("distance :", myRobot.sonar.distance_cm(), "cm")

## 4. The approach loop

The plan: look, then act on what we see. Keep going until the object is close
enough to grab (or we give up).

- `lost`   -> stop, we cannot see it.
- `left`   -> strafe left a little.
- `right`  -> strafe right a little.
- `center` -> if it is close, stop and grab; otherwise edge forward.

**Tune `GRAB_DISTANCE_CM` and the move times** so the robot ends up right over
the object before it grabs.

In [ ]:
GRAB_DISTANCE_CM = 12     # how close before we grab (tune this!)
MAX_TRIES = 20

grabbed = False
for attempt in range(MAX_TRIES):
    decision = myRobot.vision.target_position(TARGET_COLOUR, deadzone=50, show=False)
    direction = decision["direction"]
    distance = myRobot.sonar.distance_cm()
    print(attempt, direction, distance)

    if direction == "lost":
        print("Lost the object - stopping.")
        break
    elif direction == "left":
        myRobot.move.left(seconds=0.15, speed=70)
    elif direction == "right":
        myRobot.move.right(seconds=0.15, speed=70)
    elif direction == "center":
        if distance is not None and distance <= GRAB_DISTANCE_CM:
            myRobot.stop()
            if has_claw:
                grabbed = myRobot.arm.grab()
                print("Grabbed!" if grabbed else "Grab failed.")
            else:
                print("Close enough to grab - but this robot has no claw.")
            break
        else:
            myRobot.move.forward(seconds=0.3, speed=70)
    time.sleep(0.2)

myRobot.stop()
print("Done. grabbed =", grabbed)

## 5. Carry it away and put it down

In [ ]:
if grabbed:
    myRobot.move.forward(seconds=1.0, speed=80)
    myRobot.arm.place()
    print("Delivered.")
else:
    print("Nothing to deliver.")

## Challenge & tuning
- Tune `GRAB_DISTANCE_CM` until the claw lands right on the object.
- If the robot stops too far away, lower it; too close (pushes the object away),
  raise it.
- Make the strafe steps smaller for finer aim, or bigger to line up faster.
- **Advanced:** instead of sonar, a future `myRobot.vision.locate_on_floor()`
  could give the exact forward/sideways distance in cm for a one-shot approach.

In [ ]:
myRobot.stop()